 Cell 1 Setup & Load Saved W's

In [ ]:
!pip install -q chess zstandard requests
!apt-get install -y stockfish -q > /dev/null 2>&1

import math
import random
import statistics
import json
import os
import io
import time
import subprocess
import urllib.request
import re
import numpy as np
from IPython.display import SVG, display

import chess
import chess.engine
import chess.svg
import chess.pgn
import pandas as pd
import zstandard as zstd
import requests
from google.colab import drive

drive.mount('/content/drive')

RESULTS_DIR = "/content/drive/MyDrive/babygpt_results"
REASONABLE_MAX_PLIES = 400  # max plies per generated game before giving up

# Create output folder hierarchy in Drive (same structure as before)
folders = [
    "visualizations", "games", "k_table", "elo",
    "interesting_score", "repetition", "confidence_and_illegal_rate", "temperature_sweep"
]
for f in folders:
    os.makedirs(os.path.join(RESULTS_DIR, f), exist_ok=True)

# --- Install and start Ollama, in place of loading the BabyGPT checkpoint ---
!curl -fsSL https://ollama.com/install.sh | sh

ollama_process = subprocess.Popen(["ollama", "serve"])
time.sleep(5)  # give the server a moment to come up

OLLAMA_MODEL = "llama3.2:1b"
!ollama pull {OLLAMA_MODEL}

STOCKFISH_PATH = "/usr/games/stockfish"
print(f"Setup complete. Using Ollama model: {OLLAMA_MODEL}")

Cell 2: evaluation




In [ ]:
OLLAMA_URL = "http://localhost:11434/api/generate"

# ============================================================
#  Verification Loop (generic -- unchanged from before)
# ============================================================
def generate_move_with_verification(propose_move_fn, board, max_attempts=5):
    feedback = None
    for attempt in range(max_attempts):
        move_str = propose_move_fn(board, attempt, feedback)
        try:
            move = board.parse_uci(move_str)
        except ValueError:
            move = None

        if move is not None and move in board.legal_moves:
            return move, attempt + 1

        legal_sample = [board.san(m) for m in list(board.legal_moves)[:10]]
        feedback = f"'{move_str}' was not a legal move. Some legal options: {legal_sample}."
    return random.choice(list(board.legal_moves)), max_attempts


def propose_move_ollama(board, attempt, feedback, temperature=0.8, model_name=OLLAMA_MODEL):
    """Ollama-based move proposer -- same (board, attempt, feedback) -> move_str
    contract as the old propose_move_babygpt_unmasked, so it drops into
    generate_move_with_verification unchanged."""
    move_history = " ".join(m.uci() for m in board.move_stack)
    prompt = (
        f"You are playing chess. Respond with ONLY a single move in UCI "
        f"notation (e.g. e2e4), nothing else -- no explanation.\n\n"
        f"Moves played so far: {move_history if move_history else '(none, this is the first move)'}\n"
        f"FEN: {board.fen()}\n"
    )
    if feedback:
        prompt += f"\nNote: {feedback}\n"
    prompt += "\nYour move (UCI notation only):"

    response = requests.post(OLLAMA_URL, json={
        "model": model_name,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": temperature},
    })
    raw_text = response.json().get("response", "").strip()

    match = re.search(r"\b([a-h][1-8][a-h][1-8][qrbn]?)\b", raw_text.lower())
    return match.group(1) if match else "0000"  # deliberately illegal -> triggers a retry


def generate_game_with_ollama(model_name=OLLAMA_MODEL, max_new_moves=REASONABLE_MAX_PLIES,
                               max_attempts=5, temperature=0.8):
    """Replaces generate_game_with_verification_loop. Same return shape:
    (token_ids_placeholder, board, illegal_rate) -- token_ids is None since
    Ollama doesn't use the BabyGPT vocab; every downstream function here
    already works from `board`, not token ids."""
    board = chess.Board()
    illegal_attempts_total = 0
    moves_played = 0

    for _ in range(max_new_moves):
        if board.is_game_over(claim_draw=True):
            break

        move, attempts_used = generate_move_with_verification(
            propose_move_fn=lambda b, a, fb: propose_move_ollama(b, a, fb, temperature=temperature, model_name=model_name),
            board=board,
            max_attempts=max_attempts,
        )
        illegal_attempts_total += (attempts_used - 1)
        moves_played += 1
        board.push(move)

    illegal_rate = illegal_attempts_total / max(moves_played, 1)
    return None, board, illegal_rate


def generate_from_prefix_with_ollama(prefix_moves, model_name=OLLAMA_MODEL,
                                      max_new_moves=REASONABLE_MAX_PLIES,
                                      max_attempts=5, temperature=0.8):
    """Replaces generate_legal_game_from_prefix. Plays the real prefix
    moves first (no LLM calls needed), then continues via Ollama."""
    board = chess.Board()
    for move in prefix_moves:
        if move in board.legal_moves:
            board.push(move)
        else:
            break

    for _ in range(max_new_moves):
        if board.is_game_over(claim_draw=True):
            break
        move, _ = generate_move_with_verification(
            propose_move_fn=lambda b, a, fb: propose_move_ollama(b, a, fb, temperature=temperature, model_name=model_name),
            board=board,
            max_attempts=max_attempts,
        )
        board.push(move)

    return None, board


# ============================================================
#  Move Accuracy Evaluator (Ollama version)
# ============================================================
def compute_prefix_accuracy_ollama(ground_truth_moves, k, model_name=OLLAMA_MODEL,
                                    max_eval_plies=15):
    """
    Replaces compute_prefix_accuracy. BabyGPT's version checked its
    top-1 PREDICTED move against the real move at every one of the first
    k plies using its own logits -- Ollama has no equivalent internal
    top-1 concept, so this asks Ollama directly "what's your move" at
    each ground-truth position and string-compares to what was actually
    played.

    COST WARNING: this is one Ollama call PER PLY checked, per game, per
    K. Checking all k plies for large K (e.g. k=150) across 100 held-out
    games would be extremely slow -- max_eval_plies caps how many of the
    first k plies actually get checked (default 15) so this stays
    practical. Raise it if you have the time budget.
    """
    if k == 0 or len(ground_truth_moves) == 0:
        return 0.0

    eval_plies = min(k, len(ground_truth_moves), max_eval_plies)
    board = chess.Board()
    correct_predictions = 0

    for i in range(eval_plies):
        target_move = ground_truth_moves[i]
        proposed_str = propose_move_ollama(board, attempt=0, feedback=None, model_name=model_name)
        try:
            proposed_move = board.parse_uci(proposed_str)
        except ValueError:
            proposed_move = None

        if proposed_move == target_move:
            correct_predictions += 1

        board.push(target_move)

    return (correct_predictions / eval_plies) * 100.0


# ============================================================
#  Scaled Game Quality Metrics (unchanged -- works on `board` only,
#  doesn't care what generated the moves)
# ============================================================
def analyze_game_interestingness(board):
    if len(board.move_stack) < 2:
        return 100.0, 0.0

    try:
        engine = chess.engine.SimpleEngine.popen_uci(STOCKFISH_PATH)
    except Exception:
        return 100.0, 0.0

    temp_board = chess.Board()
    cp_evals = []

    for move in board.move_stack:
        temp_board.push(move)
        info = engine.analyse(temp_board, chess.engine.Limit(depth=5))
        score = info["score"].relative

        if score.is_mate():
            mate_moves = score.mate()
            if mate_moves is not None:
                cp_evals.append(1000.0 if mate_moves > 0 else -1000.0)
            else:
                cp_evals.append(0.0)
        else:
            cp_val = score.score()
            if cp_val is not None:
                cp_evals.append(float(max(-1000, min(1000, cp_val))))
            else:
                cp_evals.append(0.0)

    engine.quit()

    sharpness_raw = statistics.stdev(cp_evals) if len(cp_evals) > 1 else 0.0
    eval_swings = sum(abs(cp_evals[i] - cp_evals[i-1]) for i in range(1, len(cp_evals)))
    avg_swing = eval_swings / max(len(cp_evals) - 1, 1)

    raw_interest = (sharpness_raw * 0.4) + (avg_swing * 0.6)
    interestingness = 100.0 * (1.0 - math.exp(-raw_interest / 350.0))
    sharpness_scaled = 100.0 * math.exp(-sharpness_raw / 300.0)

    return sharpness_scaled, interestingness


# ============================================================
#  Repetition Analysis (kept -- pure board state, no model access
#  needed). Confidence is DROPPED here, see note below.
# ============================================================
def analyze_game_repetition(board):
    """
    Split out from the old analyze_game_repetition_and_confidence: the
    repetition-tracking half needs only board state, so it's unchanged.
    The confidence half doesn't carry over -- Ollama's standard API
    doesn't expose per-token probabilities the way local model logits
    do, so there's no clean equivalent to "the model's own probability
    for the move it chose." Dropped rather than faked with something
    not actually comparable.
    """
    board_replay = chess.Board()
    position_counts = {board_replay.fen().split(' ')[0]: 1}
    stuck_in_loop = False

    for move in board.move_stack:
        board_replay.push(move)
        board_key = board_replay.fen().split(' ')[0]
        position_counts[board_key] = position_counts.get(board_key, 0) + 1
        if position_counts[board_key] >= 3:
            stuck_in_loop = True

    ended_by_threefold = board_replay.can_claim_threefold_repetition()
    return stuck_in_loop, ended_by_threefold


# ============================================================
#  K-Move Prefix (Ollama version) -- alias for clarity at call sites
# ============================================================
generate_legal_game_from_prefix_ollama = generate_from_prefix_with_ollama


# ============================================================
#  Fixed Stockfish Matches with Rating (Ollama version)
# ============================================================
def play_against_stockfish_ollama(elo_rating=1500, temperature=0.8, model_is_white=True,
                                   model_name=OLLAMA_MODEL):
    engine = chess.engine.SimpleEngine.popen_uci(STOCKFISH_PATH)
    calibrated_elo = max(1320, elo_rating)
    engine.configure({"UCI_LimitStrength": True, "UCI_Elo": calibrated_elo})

    board = chess.Board()

    while not board.is_game_over(claim_draw=True):
        if (board.turn == chess.WHITE and model_is_white) or (board.turn == chess.BLACK and not model_is_white):
            move, _ = generate_move_with_verification(
                propose_move_fn=lambda b, a, fb: propose_move_ollama(b, a, fb, temperature=temperature, model_name=model_name),
                board=board,
                max_attempts=5,
            )
            board.push(move)
        else:
            result = engine.play(board, chess.engine.Limit(time=0.1))
            board.push(result.move)

    engine.quit()

    outcome = board.outcome(claim_draw=True)
    if outcome is None or outcome.winner is None:
        score = 0.5
    elif (outcome.winner == chess.WHITE and model_is_white) or (outcome.winner == chess.BLACK and not model_is_white):
        score = 1.0
    else:
        score = 0.0

    return score, board


def estimate_ollama_elo(target_elo=1500, num_games=4, temperature=0.8, model_name=OLLAMA_MODEL):
    total_score = 0.0
    for i in range(num_games):
        is_white = (i % 2 == 0)
        score, _ = play_against_stockfish_ollama(
            elo_rating=target_elo, temperature=temperature, model_is_white=is_white, model_name=model_name
        )
        total_score += score

    win_rate = total_score / num_games
    if win_rate == 0:
        performance_elo = target_elo - 400
    elif win_rate == 1:
        performance_elo = target_elo + 400
    else:
        performance_elo = target_elo - 400 * math.log10((1 / win_rate) - 1)

    return total_score, performance_elo


# ============================================================
#  Held-Out Dataset Setup (unchanged -- model-independent)
# ============================================================
HOLDOUT_PGN_URL = "https://database.lichess.org/standard/lichess_db_standard_rated_2016-08.pgn.zst"
HOLDOUT_LOCAL_ZST_PATH = "holdout_games.pgn.zst"
HOLDOUT_GAMES_PATH = f"{RESULTS_DIR}/games/holdout_games_ollama.json"
NUM_HOLDOUT_GAMES = 100
HOLDOUT_MIN_PLIES = 160
HOLDOUT_MIN_ELO = 1500

def is_good_time_control(game, min_base_seconds=180):
    tc = game.headers.get("TimeControl", "-")
    if tc == "-":
        return False
    try:
        base_seconds = int(tc.split("+")[0])
    except ValueError:
        return False
    return base_seconds >= min_base_seconds

def download_and_fix_holdout_games(num_games=NUM_HOLDOUT_GAMES, min_plies=HOLDOUT_MIN_PLIES):
    if os.path.exists(HOLDOUT_GAMES_PATH):
        print(f"Loading already-fixed held-out games from {HOLDOUT_GAMES_PATH}")
        with open(HOLDOUT_GAMES_PATH, "r") as f:
            saved = json.load(f)
        return [[chess.Move.from_uci(u) for u in g["moves_uci"]] for g in saved]

    print(f"Downloading held-out PGN archive: {HOLDOUT_PGN_URL}")
    urllib.request.urlretrieve(HOLDOUT_PGN_URL, HOLDOUT_LOCAL_ZST_PATH)

    games = []
    games_to_save = []
    dctx = zstd.ZstdDecompressor()
    with open(HOLDOUT_LOCAL_ZST_PATH, "rb") as fh:
        with dctx.stream_reader(fh) as reader:
            text_stream = io.TextIOWrapper(reader, encoding="utf-8")
            while len(games) < num_games:
                game = chess.pgn.read_game(text_stream)
                if game is None:
                    break
                try:
                    white_elo = int(game.headers.get("WhiteElo", 0))
                    black_elo = int(game.headers.get("BlackElo", 0))
                except ValueError:
                    continue
                if white_elo < HOLDOUT_MIN_ELO or black_elo < HOLDOUT_MIN_ELO:
                    continue
                if not is_good_time_control(game):
                    continue
                if game.headers.get("Termination", "") != "Normal":
                    continue

                moves = list(game.mainline_moves())
                if len(moves) < min_plies:
                    continue

                games.append(moves)
                games_to_save.append({"moves_uci": [m.uci() for m in moves]})

    os.makedirs(os.path.dirname(HOLDOUT_GAMES_PATH), exist_ok=True)
    with open(HOLDOUT_GAMES_PATH, "w") as f:
        json.dump(games_to_save, f, indent=2)
    print(f"Fixed {len(games)} held-out games, saved to {HOLDOUT_GAMES_PATH}")
    return games


def run_heldout_k_experiment_ollama(holdout_games, k_values, temperature, model_name=OLLAMA_MODEL):
    interest_data = {}
    accuracy_data = {}

    for k in k_values:
        k_interest_scores = []
        k_accuracies = []

        for game_moves in holdout_games:
            prefix = [] if k == 0 else game_moves[:min(k, len(game_moves))]

            acc = compute_prefix_accuracy_ollama(game_moves, k, model_name=model_name)
            k_accuracies.append(acc)

            _, final_board = generate_from_prefix_with_ollama(
                prefix, model_name=model_name, temperature=temperature
            )
            _, interest = analyze_game_interestingness(final_board)
            k_interest_scores.append(interest)

        mean_int = np.mean(k_interest_scores)
        sd_int = np.std(k_interest_scores, ddof=1) if len(k_interest_scores) > 1 else 0.0
        mean_acc = np.mean(k_accuracies)
        sd_acc = np.std(k_accuracies, ddof=1) if len(k_accuracies) > 1 else 0.0

        interest_data[f"K={k}"] = f"{mean_int:.1f} ± {sd_int:.1f}"
        accuracy_data[f"K={k}"] = f"{mean_acc:.1f}% ± {sd_acc:.1f}%"

        print(f"K={k}: Accuracy = {mean_acc:.1f}% ± {sd_acc:.1f}% | Interest = {mean_int:.1f} ± {sd_int:.1f}")

    return pd.DataFrame([accuracy_data, interest_data], index=["Ground-Truth Accuracy", "Interestingness Score"])

Model reloaded for inference successfully.


Cell 3: ran k sweep and analysis

In [ ]:
# Restored to your original scale. Be aware this means a LOT of live
# Ollama calls (every move, plus illegal-move retries) -- this can take
# a very long time to finish compared to the local BabyGPT model.

TEMPERATURES = [0.1, 0.3, 0.5, 0.7, 0.8, 1.0, 1.2, 1.5, 2.0]
GAMES_PER_TEMP = 8

all_temp_summaries = []

for T in TEMPERATURES:
    print(f"\n==================================================")
    print(f"       RUNNING ANALYSIS FOR TEMPERATURE T = {T}")
    print(f"==================================================")

    temp_records = []

    for g in range(1, GAMES_PER_TEMP + 1):
        print(f"\n--- [T={T}] Generating Game {g}/{GAMES_PER_TEMP} ---")

        _, board, illegal_rate = generate_game_with_ollama(
            max_new_moves=REASONABLE_MAX_PLIES, max_attempts=5, temperature=T
        )

        stuck, threefold = analyze_game_repetition(board)
        sharpness, interest = analyze_game_interestingness(board)

        status = "COMPLETED" if board.is_game_over(claim_draw=True) else "TRUNCATED"
        print(f"Game {g} Status: {status} | Moves: {len(board.move_stack)} | Result: {board.result(claim_draw=True)}")
        print(f"Metrics -> Illegal Rate: {illegal_rate:.4f} | Interestingness (0-100): {interest:.2f} | Sharpness (Stability): {sharpness:.2f}")

        last_move = board.peek() if len(board.move_stack) > 0 else None
        svg_data = chess.svg.board(board=board, lastmove=last_move)
        svg_filename = f"{RESULTS_DIR}/visualizations/ollama_temp_{T}_game_{g}_{status.lower()}.svg"
        with open(svg_filename, "w") as f_svg:
            f_svg.write(svg_data)

        temp_records.append({
            "temperature": T,
            "game_id": g,
            "moves": len(board.move_stack),
            "illegal_rate": illegal_rate,
            "stuck_loop": stuck,
            "threefold_end": threefold,
            "sharpness": sharpness,
            "interestingness": interest,
        })

    print(f"\n--- Running Multi-Game Stockfish Match for T={T} ---")
    score_total, est_elo = estimate_ollama_elo(target_elo=1500, num_games=4, temperature=T)
    print(f"Stockfish Match Points: {score_total}/4.0 | Estimated Performance Elo: {est_elo:.0f}")

    df_temp = pd.DataFrame(temp_records)
    avg_len = df_temp["moves"].mean()
    avg_illegal = df_temp["illegal_rate"].mean()
    avg_interest = df_temp["interestingness"].mean()
    truncation_rate = (df_temp["moves"] == REASONABLE_MAX_PLIES).mean() * 100.0

    print(f"\n--- TEMPERATURE {T} SUMMARY ---")
    print(f"Avg Game Length: {avg_len:.1f} moves")
    print(f"Truncation Rate: {truncation_rate:.1f}%")
    print(f"Avg Illegal-Move Rate: {avg_illegal:.4f}")
    print(f"Avg Interestingness (0-100): {avg_interest:.2f}")

    summary_entry = {
        "temperature": T,
        "avg_game_length": avg_len,
        "truncation_rate_pct": truncation_rate,
        "avg_illegal_rate": avg_illegal,
        "avg_interestingness": avg_interest,
        "stockfish_match_score": score_total,
        "estimated_elo": est_elo,
    }
    all_temp_summaries.append(summary_entry)

    with open(f"{RESULTS_DIR}/temperature_sweep/ollama_temp_{T}_full_results.json", "w") as f_json:
        json.dump(temp_records, f_json, indent=4)

df_all = pd.DataFrame(all_temp_summaries)
df_all.to_csv(f"{RESULTS_DIR}/full_temperature_sweep_summary_ollama.csv", index=False)

print("\n==================================================")
print("     FINAL COMPARISON ACROSS ALL TEMPERATURES")
print("==================================================")
print(df_all.to_string(index=False))

best_row = df_all.loc[df_all["avg_interestingness"].idxmax()]
BEST_TEMPERATURE = best_row["temperature"]
print(f"\nBest temperature by avg interestingness: T={BEST_TEMPERATURE} "
      f"(avg interestingness={best_row['avg_interestingness']:.2f})")

# --- Held-Out K-Prefix Sweep with Accuracy Calculation ---
holdout_games = download_and_fix_holdout_games(num_games=100)

K_VALUES_HELDOUT = [10, 25, 50, 100, 150]
print(f"\n--- Running Held-Out K-Prefix Experiment at T={BEST_TEMPERATURE} "
      f"({len(holdout_games)} fixed real games, never seen in training) ---")

heldout_k_df = run_heldout_k_experiment_ollama(
    holdout_games, k_values=K_VALUES_HELDOUT, temperature=BEST_TEMPERATURE
)

print("\n=== HELD-OUT K-PREFIX METRICS (mean ± sd) ===")
print(heldout_k_df.to_string())

heldout_k_df.to_csv(f"{RESULTS_DIR}/k_table/heldout_k_sweep_best_temperature_ollama.csv")
print(f"\nSaved to {RESULTS_DIR}/k_table/heldout_k_sweep_best_temperature_ollama.csv")

Cell 4: Run Games and Analysis

In [ ]:
TEMPERATURES = [0.1, 0.3, 0.5, 0.7, 0.8, 1.0, 1.2, 1.5, 2.0]
GAMES_PER_TEMP = 8

all_temp_summaries = []

for T in TEMPERATURES:
    print(f"\n==================================================")
    print(f"       RUNNING ANALYSIS FOR TEMPERATURE T = {T}")
    print(f"==================================================")

    temp_records = []

    for g in range(1, GAMES_PER_TEMP + 1):
        print(f"\n--- [T={T}] Generating Game {g}/{GAMES_PER_TEMP} ---")

        token_ids, board, illegal_rate = generate_game_with_verification_loop(
            model, max_new_moves=REASONABLE_MAX_PLIES, max_attempts=5
        )

        stuck, threefold, avg_conf = analyze_game_repetition_and_confidence(model, token_ids)
        sharpness, interest = analyze_game_interestingness(board)

        status = "COMPLETED" if board.is_game_over(claim_draw=True) else "TRUNCATED"
        print(f"Game {g} Status: {status} | Moves: {len(board.move_stack)} | Result: {board.result(claim_draw=True)}")
        print(f"Metrics -> Confidence: {avg_conf:.4f} | Illegal Rate: {illegal_rate:.4f} | Interestingness (0-100): {interest:.2f} | Sharpness (Stability): {sharpness:.2f}")

        last_move = board.peek() if len(board.move_stack) > 0 else None
        svg_data = chess.svg.board(board=board, lastmove=last_move)
        svg_filename = f"{RESULTS_DIR}/visualizations/temp_{T}_game_{g}_{status.lower()}.svg"
        with open(svg_filename, "w") as f_svg:
            f_svg.write(svg_data)

        temp_records.append({
            "temperature": T,
            "game_id": g,
            "moves": len(board.move_stack),
            "illegal_rate": illegal_rate,
            "stuck_loop": stuck,
            "threefold_end": threefold,
            "confidence": avg_conf,
            "sharpness": sharpness,
            "interestingness": interest
        })

    print(f"\n--- Running Multi-Game Stockfish Match for T={T} ---")
    score_total, est_elo = estimate_model_elo(model, target_elo=1500, num_games=4, temperature=T)
    print(f"Stockfish Match Points: {score_total}/4.0 | Estimated Performance Elo: {est_elo:.0f}")

    df_temp = pd.DataFrame(temp_records)
    avg_len = df_temp["moves"].mean()
    avg_illegal = df_temp["illegal_rate"].mean()
    avg_conf = df_temp["confidence"].mean()
    avg_interest = df_temp["interestingness"].mean()
    truncation_rate = (df_temp["moves"] == REASONABLE_MAX_PLIES).mean() * 100.0

    print(f"\n--- TEMPERATURE {T} SUMMARY ---")
    print(f"Avg Game Length: {avg_len:.1f} moves")
    print(f"Truncation Rate: {truncation_rate:.1f}%")
    print(f"Avg Confidence: {avg_conf:.4f}")
    print(f"Avg Illegal-Move Rate: {avg_illegal:.4f}")
    print(f"Avg Interestingness (0-100): {avg_interest:.2f}")

    summary_entry = {
        "temperature": T,
        "avg_game_length": avg_len,
        "truncation_rate_pct": truncation_rate,
        "avg_confidence": avg_conf,
        "avg_illegal_rate": avg_illegal,
        "avg_interestingness": avg_interest,
        "stockfish_match_score": score_total,
        "estimated_elo": est_elo
    }
    all_temp_summaries.append(summary_entry)

    with open(f"{RESULTS_DIR}/temperature_sweep/temp_{T}_full_results.json", "w") as f_json:
        json.dump(temp_records, f_json, indent=4)

df_all = pd.DataFrame(all_temp_summaries)
df_all.to_csv(f"{RESULTS_DIR}/full_temperature_sweep_summary.csv", index=False)

print("\n==================================================")
print("     FINAL COMPARISON ACROSS ALL TEMPERATURES")
print("==================================================")
print(df_all.to_string(index=False))

best_row = df_all.loc[df_all["avg_interestingness"].idxmax()]
BEST_TEMPERATURE = best_row["temperature"]
print(f"\nBest temperature by avg interestingness: T={BEST_TEMPERATURE} "
      f"(avg interestingness={best_row['avg_interestingness']:.2f})")

# Run Held-Out K-Prefix Sweep with Accuracy Calculation
holdout_games = download_and_fix_holdout_games()

K_VALUES_HELDOUT = [10, 25, 50, 100, 150]
print(f"\n--- Running Held-Out K-Prefix Experiment at T={BEST_TEMPERATURE} "
      f"({len(holdout_games)} fixed real games, never seen in training) ---")

heldout_k_df = run_heldout_k_experiment(
    model, holdout_games, k_values=K_VALUES_HELDOUT, temperature=BEST_TEMPERATURE
)

print("\n=== HELD-OUT K-PREFIX METRICS (mean \u00b1 sd) ===")
print(heldout_k_df.to_string())

heldout_k_df.to_csv(f"{RESULTS_DIR}/k_table/heldout_k_sweep_best_temperature.csv")
print(f"\nSaved to {RESULTS_DIR}/k_table/heldout_k_sweep_best_temperature.csv")


       RUNNING ANALYSIS FOR TEMPERATURE T = 0.1

--- [T=0.1] Generating Game 1/8 ---
Game 1 Status: COMPLETED | Moves: 287 | Result: 1/2-1/2
Metrics -> Confidence: 0.0710 | Illegal Rate: 2.4530 | Interestingness (0-100): 79.27 | Sharpness (Stability): 23.78

--- [T=0.1] Generating Game 2/8 ---
Game 2 Status: COMPLETED | Moves: 361 | Result: 1/2-1/2
Metrics -> Confidence: 0.0878 | Illegal Rate: 2.0526 | Interestingness (0-100): 89.52 | Sharpness (Stability): 13.90

--- [T=0.1] Generating Game 3/8 ---
Game 3 Status: TRUNCATED | Moves: 400 | Result: *
Metrics -> Confidence: 0.0720 | Illegal Rate: 2.3625 | Interestingness (0-100): 82.47 | Sharpness (Stability): 20.03

--- [T=0.1] Generating Game 4/8 ---
Game 4 Status: COMPLETED | Moves: 203 | Result: 1/2-1/2
Metrics -> Confidence: 0.0911 | Illegal Rate: 1.9261 | Interestingness (0-100): 94.50 | Sharpness (Stability): 10.47

--- [T=0.1] Generating Game 5/8 ---
Game 5 Status: TRUNCATED | Moves: 400 | Result: *
Metrics -> Confidence: 0.0911 